In [ ]:
# Libraries

import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Read data from one of the datasets

df = pd.read_excel(path)

In [ ]:
# Feature engineering - "headers" column

start_time_headers1 = time.time()
def json_to_dict(json_str):
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        return {}

df_dicts = df['headers'].apply(json_to_dict)

df_expanded = pd.json_normalize(df_dicts)
end_time_headers1 = time.time()

# Identify and remove columns with many empty lines
threshold = 0.02*len(df_expanded)
columns_to_drop = [col for col in df_expanded.columns if df_expanded[col].isna().sum() > threshold]
start_time_headers2 = time.time()
df_cleaned = df_expanded.drop(columns=columns_to_drop)

# Concatenate the original Dataframe and the expanded Dataframe from "headers" column
df = pd.concat([df, df_cleaned], axis=1)

# Remove "headers" original column
df.drop(columns=['headers'], inplace=True)
end_time_headers2 = time.time()

In [ ]:
# Feature engineering - "request" column
start_time_request1 = time.time()
def json_to_dict(json_str):
    if isinstance(json_str, str):
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return {}
    return {}

df_dicts = df['request'].apply(json_to_dict)
df_dicts.head(len(df_dicts))

df_expanded = pd.json_normalize(df_dicts)
end_time_request1 = time.time()

# Select columns with practical significance
request_columns = ['a', 'b', 'c', 'd', 'e']
start_time_request2 = time.time()
cols_to_keep = request_columns
df_filtered = df_expanded.loc[:, cols_to_keep]
end_time_request2 = time.time()

# Concatenate the original Dataframe and the expanded Dataframe from "request" column
df = pd.concat([df, df_filtered], axis=1)

# Remove "request" original column
df=df.drop(columns=['request'])

In [ ]:
# Change "OK", "Observation" and "Fraud" for 0, 1 and 2 respectively in "decision" column, which is the target column
# Create a mapping dictionary
decision_map = {'OK': 0, 'Observation': 1, 'Fraud': 2}

# Replace values ​​in the 'decision' column using the map method
df['decision'] = df['decision'].map(decision_map)

In [ ]:
# Treatment time of the request columns and headers per sample
time_headers_request = (end_time_headers1 - start_time_headers1 + end_time_headers2 - start_time_headers2 + end_time_request1 - start_time_request1 + end_time_request2 - start_time_request2) / len(df)

In [ ]:
# Separate Dataframe into input data (features) and output data (labels)
X = df.drop(columns=['decision'])
y = df['decision']

In [ ]:
# Display the absolute count and percentage of each class
class_distribution = df['decision'].value_counts()
class_percentage = df['decision'].value_counts(normalize=True) * 100

class_distribution_results = pd.DataFrame({
    'Count': class_distribution,
    'Percentage (%)': pclass_percentage
})

print(class_distribution_results)


In [ ]:
# Stratified K-Fold

# Instantiate StratifiedKFold
skf = StratifiedKFold(n_splits=10)

# To store the results of each fold
accuracies = []
balanced_accuracies = []
recalls = []
total_time = []
precisions = []
f1_scores = []
roc_aucs_by_class = []
micro_precisions = []
micro_recalls = []
micro_f1_scores = []
micro_roc_aucs = []

# Loop over splits
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    processing_time = []

    # Natural Language Processing in each train column
    for col in X_train.columns:
        # Get unique column values
        unique_values = X_train[col].unique()

        # Check if all values ​​are empty or NaN
        if len(unique_values) == 1 and (unique_values[0] == '' or pd.isna(unique_values[0])):
            print(f"Column '{col}' was ignored due to empty values ​​or NaN.")
            X_train=X_train.drop(columns=[col])
            X_test=X_test.drop(columns=[col])
            continue  # Ignore this column and move to the next
        elif len(unique_values) == 2 and (unique_values[0] == '' or pd.isna(unique_values[0])) and (unique_values[1] == '' or pd.isna(unique_values[1])):
            print(f"Column '{col}' was ignored due to empty values ​​or NaN.")
            X_train=X_train.drop(columns=[col])
            X_test=X_test.drop(columns=[col])
            continue  # Ignore this column and move to the next

        # Initial column treatment
        X_train.loc[:, col] = X_train[col].fillna('').astype(str).str.replace(r'[^\w\s]', ' ', regex=True)
        start_time0 = time.time()
        X_test.loc[:, col] = X_test[col].fillna('').astype(str).str.replace(r'[^\w\s]', ' ', regex=True)
        end_time0 = time.time()

        # Instatiate TextVectorization
        vectorizer = tf.keras.layers.TextVectorization(output_mode='int', standardize=None, output_sequence_length=30)
        vectorizer.adapt(X_train[col].values)  # Vector adjustment to training vocabulary

        # Convert text into tokens sequences with automatic padding
        train_padded = vectorizer(X_train[col].values)
        start_time1 = time.time()
        test_padded = vectorizer(X_test[col].values)
        end_time1 = time.time()

        # Convert to dataframe and replace the original columns
        X_train_col = pd.DataFrame(train_padded.numpy(), index=X_train.index)
        X_train = X_train.drop(columns=[col])
        X_train = pd.concat([X_train, X_train_col], axis=1)

        start_time2 = time.time()
        X_test_col = pd.DataFrame(test_padded.numpy(), index=X_test.index)
        X_test = X_test.drop(columns=[col])
        X_test = pd.concat([X_test, X_test_col], axis=1)
        end_time2 = time.time()
        processing_time.append((end_time0 - start_time0) + (end_time1 - start_time1) + (end_time2 - start_time2))

    # Decision Tree
    X_train.columns = X_train.columns.astype(str)
    start_time3 = time.time()
    X_test.columns = X_test.columns.astype(str)
    end_time3 = time.time()

    model = DecisionTreeClassifier()
    model.fit(X_train, y_train) # training

    # Predict method on the test set
    start_time4 = time.time()
    y_pred = model.predict(X_test)
    end_time4 = time.time()

    # Balanced Accuracy
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
    balanced_accuracies.append(balanced_accuracy)

    # Recall
    recall = recall_score(y_test, y_pred, average=None)
    recalls.append(recall)

    micro_recall = recall_score(y_test, y_pred, average='micro')
    micro_recalls.append(micro_recall)

    # Precision
    precision = precision_score(y_test, y_pred, average=None)
    precisions.append(precision)

    micro_precision = precision_score(y_test, y_pred, average='micro')
    micro_precisions.append(micro_precision)

    # F1-Score

    f1 = f1_score(y_test, y_pred, average=None)
    f1_scores.append(f1)

    micro_f1 = f1_score(y_test, y_pred, average='micro')
    micro_f1_scores.append(micro_f1)

    # Calculate ROC-AUC for each class separately
    roc_auc_classes = []
    y_proba = model.predict_proba(X_test)
    for class_idx in range(len(model.classes_)):
        y_true_binary = (y_test == class_idx).astype(int)
        auc = roc_auc_score(y_true_binary, y_proba[:, class_idx])
        roc_auc_classes.append(auc)
    roc_aucs_by_class.append(roc_auc_classes)

    micro_roc_auc = roc_auc_score(y_test, model.predict_proba(X_test), multi_class="ovr", average="micro")
    micro_roc_aucs.append(micro_roc_auc)

    total_time.append(sum(tempo_processamento) + end_time3-start_time3 + end_time4-start_time4)


# Calculate average and standard deviation for each metric between 10 folds
mean_balanced_accuracy = np.mean(balanced_accuracies)
std_balanced_accuracy = np.std(balanced_accuracies)

mean_recall = np.mean(recalls, axis=0)
std_recall = np.std(recalls, axis=0)

mean_micro_recall = np.mean(micro_recalls)
std_micro_recall = np.std(micro_recalls)

mean_precision = np.mean(precisions, axis=0)
std_precision = np.std(precisions, axis=0)

mean_micro_precision = np.mean(micro_precisions)
std_micro_precision = np.std(micro_precisions)

mean_f1 = np.mean(f1_scores, axis=0)
std_f1 = np.std(f1_scores, axis=0)

mean_micro_f1 = np.mean(micro_f1_scores)
std_micro_f1 = np.std(micro_f1_scores)

roc_aucs_by_class = np.array(roc_aucs_by_class)
mean_roc_auc_by_class = np.mean(roc_aucs_by_class, axis=0)
std_roc_auc_by_class = np.std(roc_aucs_by_class, axis=0)

mean_micro_roc_auc = np.mean(micro_roc_aucs)
std_micro_roc_auc = np.std(micro_roc_aucs)

time_per_sample = (np.mean(total_time) / len(X_test))*1000 + tempo*1000

print(f"Average Sample Processing Time: {time_per_sample:.4f} ms")
print()

print(f"Balanced accuracy average: {mean_balanced_accuracy}")
print(f"Standard deviation of balanced accuracy: {std_balanced_accuracy}")
print()

print(f"Average recall for each class: {mean_recall}")
print(f"Standard deviation of recall for each class: {std_recall}")
print()

print(f"Average micro-recall: {mean_micro_recall}")
print(f"Standard deviation of micro-recall: {std_micro_recall}")
print()

print(f"Average precision for each class: {mean_precision}")
print(f"Standard deviation of precision for each class: {std_precision}")
print()

print(f"Average micro-precision: {mean_micro_precision}")
print(f"Standard deviation of micro-precision: {std_micro_precision}")
print()

print(f"Average F1-Score for each class: {mean_f1}")
print(f"Standard deviation of F1-Score for eacho class: {std_f1}")
print()

print(f"Average micro-F1-Score: {mean_micro_f1}")
print(f"Standard deviation of micro-F1-Score: {std_micro_f1}")
print()

for class_idx, (mean_auc, std_auc) in enumerate(zip(mean_roc_auc_by_class, std_roc_auc_by_class)):
    print(f"Class {class_idx}: Average ROC-AUC = {mean_auc:.4f}, standard deviation of ROC_AUC = {std_auc:.4f}")

print(f"Average micro-ROC-AUC: {mean_micro_roc_auc}")
print(f"Standard deviation of micro-ROC-AUC: {std_micro_roc_auc}")
print()


In [ ]:
# Grid search loop for decision tree classifier with Stratified K-Fold

# Hyperparameter lists
criterion_list =  ['gini', 'entropy', 'log_loss']
splitter_list = ['best', 'random']
max_features_list = ['sqrt', 'log2', None]
ccp_alpha_list = [0.1, .01, .001]
class_weight_list = ['balanced', None]
min_samples_split_list = [0.01, 0.1, 0.15]
min_samples_leaf_list = [0.01, 0.1, 0.15]
min_impurity_decrease_list = [0.0, 0.01, 0.05, 0.1, 0.15]
min_weight_fraction_leaf_list = [0.01, 0.05, 0.1, 0.15]

# Instantiate StratifiedKFold
skf = StratifiedKFold(n_splits=5)

# Loop over splits
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Natural Language Processing in each train column
    for col in X_train.columns:
        # Get unique column values
        unique_values = X_train[col].unique()

        # Check if all values ​​are empty or NaN
        if len(unique_values) == 1 and (unique_values[0] == '' or pd.isna(unique_values[0])):
            print(f"Column '{col}' was ignored due to empty values ​​or NaN.")
            X_train=X_train.drop(columns=[col])
            X_test=X_test.drop(columns=[col])
            continue  # Ignore this column and move to the next
        elif len(unique_values) == 2 and (unique_values[0] == '' or pd.isna(unique_values[0])) and (unique_values[1] == '' or pd.isna(unique_values[1])):
            print(f"Column '{col}' was ignored due to empty values ​​or NaN.")
            X_train=X_train.drop(columns=[col])
            X_test=X_test.drop(columns=[col])
            continue  # Ignore this column and move to the next

        # Initial column treatment
        X_train.loc[:, col] = X_train[col].fillna('').astype(str).str.replace(r'[^\w\s]', ' ', regex=True)
        X_test.loc[:, col] = X_test[col].fillna('').astype(str).str.replace(r'[^\w\s]', ' ', regex=True)

        # Instatiate TextVectorization
        vectorizer = tf.keras.layers.TextVectorization(output_mode='int', standardize=None, output_sequence_length=30)
        vectorizer.adapt(X_train[col].values)  # Vector adjustment to training vocabulary

        # Convert text into tokens sequences with automatic padding
        train_padded = vectorizer(X_train[col].values)
        test_padded = vectorizer(X_test[col].values)

        # Convert to dataframe and replace the original columns
        X_train_col = pd.DataFrame(train_padded.numpy(), index=X_train.index)
        X_train = X_train.drop(columns=[col])
        X_train = pd.concat([X_train, X_train_col], axis=1)

        X_test_col = pd.DataFrame(test_padded.numpy(), index=X_test.index)
        X_test = X_test.drop(columns=[col])
        X_test = pd.concat([X_test, X_test_col], axis=1)

    # Decision Tree
    X_train.columns = X_train.columns.astype(str)
    X_test.columns = X_test.columns.astype(str)

    for criterion in criterion_list:
      for splitter in splitter_list:
          for max_features in max_features_list:
              for ccp_alpha in ccp_alpha_list:
                  for class_weight in class_weight_list:
                      for min_samples_split in min_samples_split_list:
                          for min_samples_leaf in min_samples_leaf_list:
                              for min_impurity_decrease in min_impurity_decrease_list:
                                  for min_weight_fraction_leaf in min_weight_fraction_leaf_list:
                                      model = DecisionTreeClassifier(criterion=criterion, splitter=splitter, max_features=max_features, ccp_alpha=ccp_alpha, max_depth=None,
                                                              max_leaf_nodes=None, class_weight=class_weight, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
                                                              min_impurity_decrease=min_impurity_decrease, min_weight_fraction_leaf=min_weight_fraction_leaf)

                                      history = model.fit(X_train, y_train)

                                      # make predictions
                                      y_predicted = model.predict(X_test)

                                      # evaluate predictions
                                      # balanced accuracy
                                      balanced_accuracy = balanced_accuracy_score(y_test, y_predicted)

                                      # recall
                                      recall = recall_score(y_test, y_predicted, average=None)

                                      results_dict = {
                                          "fold" : i,
                                          "criterion": criterion,
                                          "splitter": splitter,
                                          "max_features": max_features,
                                          "ccp_alpha": ccp_alpha,
                                          "class_weight": class_weight,
                                          "min_samples_split": min_samples_split,
                                          "min_samples_leaf": min_samples_leaf,
                                          "min_impurity_decrease": min_impurity_decrease,
                                          "min_weight_fraction_leaf": min_weight_fraction_leaf,
                                          "balanced accuracy": balanced_accuracy,
                                          "recalls": recall
                                      }
                                      results.append(results_dict)

df = pd.DataFrame(results)

# Treat the results (recall and balanced accuracy) and make a new dataframe with the averages and standard deviation between the folds
num = int(df.shape[0]/5)

bal_acc = []
avgs_bal_acc = []
stds_bal_acc = []

recalls = []
avgs_recalls = []
stds_recalls = []

for n in range(0,num):
  bal_acc1 = [df["balanced accuracy"][n], df["balanced accuracy"][n+num],df["balanced accuracy"][n+(2*num)], df["balanced accuracy"][n+(3*num)], df["balanced accuracy"][n+(4*num)]]
  recall1 = [df["recalls"][n], df["recalls"][n+num],df["recalls"][n+(2*num)], df["recalls"][n+(3*num)], df["recalls"][n+(4*num)]]
  bal_acc.append(bal_acc1)
  avgs_bal_acc.append(np.mean(bal_acc[n]))
  stds_bal_acc.append(np.std(bal_acc[n]))

  recalls.append(recall1)
  avgs_recalls.append(np.mean(recalls[n], axis=0))
  stds_recalls.append(np.std(recalls[n], axis=0))


avg_std = {
    "Bal. acc.": avgs_bal_acc,
    "Bal. acc. (std)": stds_bal_acc,
}

num_classes = len(avgs_recalls[0])
for i in range(num_classes):
    avg_std[f"Recall Class {i+1}"] = [recall[i] for recall in avgs_recalls]
    avg_std[f"Recall (std) Class {i+1}"] = [std[i] for std in stds_recalls]

df_avg_std = pd.DataFrame(avg_std)

df_end = df.drop(columns=["fold", "balanced accuracy", "recalls"])
df_end = df_end.drop(range(num, num*5))
df_end = pd.concat([df_end, df_avg_std], axis=1)

# Sorting df_end according to the balanced accuracy (the biggest in the first line)
df_end = df_end.sort_values("Bal. acc.", ascending=False)

# The first line will show the best hyperparameters
